# Try to train Pycaret with entire dataset

This will be the code in order to train the model completely. I tried t run it, but my pc runs out of application memory, let alone the time that requires to train it.

Author: José Fernando Gutiérrez Montero

In [1]:
# Cell 1: Imports y configuración inicial
import pandas as pd
from pycaret.regression import *
import os

MODEL_PATH = './../../data/trained_models/housing/'
os.makedirs(MODEL_PATH, exist_ok=True)

In [2]:
# Cell 2: Cargar datos
file_path = './../../data/housing/processed/price_paid_model_ready.parquet'
df = pd.read_parquet(file_path)

# Revisar las columnas
df.columns

Index(['price', 'sale_date', 'property_type', 'old_new', 'duration',
       'town_city', 'district', 'county', 'record_status___monthly_file_only',
       'sale_year'],
      dtype='object')

In [3]:
# Cell 3: Convertimos sale_date a datetime y generamos features temporales
df['sale_date'] = pd.to_datetime(df['sale_date'])

df['Year'] = df['sale_date'].dt.year
df['Month'] = df['sale_date'].dt.month
df['Day'] = df['sale_date'].dt.day
df['Weekday'] = df['sale_date'].dt.weekday  # lunes=0

# Creamos df_model para PyCaret y quitamos la columna original de fecha
df_model = df.drop(columns=['sale_date'])

In [8]:
# Cell 4: Setup de PyCaret
reg = setup(
    data=df_model,
    target='price',
    session_id=123,
    numeric_features=['Year','Month','Day','Weekday'],
    categorical_features=['property_type','old_new','duration','town_city','district','county','record_status___monthly_file_only'],
    fold_strategy='timeseries',   # validación por años
    fold=5,
    transform_target=True,
    data_split_shuffle=False,
    fold_shuffle=False,
    verbose=True
)

,Description,Value
0,Session id,123
1,Target,price
2,Target type,Regression
3,Original data shape,"(22480822, 13)"
4,Transformed data shape,"(22480822, 17)"
5,Transformed train set shape,"(15736575, 17)"
6,Transformed test set shape,"(6744247, 17)"
7,Numeric features,4
8,Categorical features,7
9,Rows with missing values,0.0%


In [11]:
numeric_cols = df_model.select_dtypes(include='number').columns
df_model[numeric_cols].corr()['price'].sort_values(ascending=False)

price        1.000000
sale_year    0.287468
Year         0.287468
Month        0.005272
Day         -0.009884
Weekday     -0.038544
Name: price, dtype: float64

Training this with the entire dataset is crazy

In [12]:
# Cell 5: Comparar modelos y seleccionar top 3
best_model = compare_models(sort='RMSE', n_select=3)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,20:24:43
Status,. . . . . . . . . . . . . . . . . .,Fitting 5 Folds
Estimator,. . . . . . . . . . . . . . . . . .,K Neighbors Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lar,Least Angle Regression,43772.8304,12342048401.3852,96663.8527,0.2338,0.3525,0.2667,25.1240
br,Bayesian Ridge,52159.7082,14501414513.0388,117438.4677,0.2866,0.4464,0.3338,26.9880
ridge,Ridge Regression,52159.6928,14501430683.1184,117438.5369,0.2866,0.4464,0.3338,25.7560
lr,Linear Regression,52161.2439,14501582564.2064,117439.3056,0.2866,0.4464,0.3338,26.6340
omp,Orthogonal Matching Pursuit,72146.3811,21925242200.1468,144188.4996,-0.0677,0.6470,0.5291,25.3700
huber,Huber Regressor,80334.6834,23632357392.3182,148614.6586,-0.1190,0.7339,0.4543,119.2640
par,Passive Aggressive Regressor,84238.1415,24198364711.4658,150367.7266,-0.1622,0.7979,0.5415,34.5840
en,Elastic Net,84291.8535,26653669132.5660,158303.8640,-0.2783,0.7827,0.4973,26.8240
lasso,Lasso Regression,88637.1626,28099183750.4698,161558.0853,-0.3161,0.8231,0.5103,25.4140
llar,Lasso Least Angle Regression,88637.1626,28099183750.4698,161558.0853,-0.3161,0.8231,0.5103,25.6140


Processing:   0%|          | 0/83 [00:00<?, ?it/s]

KeyboardInterrupt: 